# Model 1: Vehicle Detection (Cars & Motorcycles)
**YOLOv8 training on Kaggle GPU**

This notebook trains a YOLOv8 model to detect and count:
- Cars
- Motorcycles

Update `DATASET_PATH` to point to your Kaggle dataset.

In [ ]:
# Install dependencies
!pip install ultralytics -q
!pip install roboflow -q

In [ ]:
import os
import yaml
import shutil
from pathlib import Path
from ultralytics import YOLO

# ── Configuration ──────────────────────────────────────────────
# Update this to your Kaggle dataset path
DATASET_PATH = '/kaggle/input/your-vehicle-dataset'   # <-- CHANGE THIS
PROJECT_NAME = 'vehicle_detection'
MODEL_BASE   = 'yolov8n.pt'   # yolov8n / s / m / l / x
EPOCHS       = 50
IMG_SIZE     = 640
BATCH_SIZE   = 16
# ───────────────────────────────────────────────────────────────

print('Ultralytics version:', __import__('ultralytics').__version__)

In [ ]:
# ── Option A: Dataset already has YOLO structure ────────────────
# Expected layout:
#   DATASET_PATH/
#     train/images/  train/labels/
#     val/images/    val/labels/
#     data.yaml

data_yaml = os.path.join(DATASET_PATH, 'data.yaml')

# ── Option B: Create data.yaml manually ────────────────────────
# Run this cell only if your dataset has no data.yaml
if not os.path.exists(data_yaml):
    config = {
        'path': DATASET_PATH,
        'train': 'train/images',
        'val':   'val/images',
        'nc': 2,
        'names': ['car', 'motorcycle']
    }
    data_yaml = '/kaggle/working/vehicle_data.yaml'
    with open(data_yaml, 'w') as f:
        yaml.dump(config, f)
    print('Created data.yaml at', data_yaml)
else:
    print('Found existing data.yaml')

# Print the config so you can verify class names / paths
with open(data_yaml) as f:
    print(f.read())

In [ ]:
# ── Train ───────────────────────────────────────────────────────
model = YOLO(MODEL_BASE)

results = model.train(
    data    = data_yaml,
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH_SIZE,
    project = '/kaggle/working/runs',
    name    = PROJECT_NAME,
    device  = 0,            # GPU
    patience= 15,           # early-stop patience
    save    = True,
    plots   = True,
    verbose = True,
    # Augmentation
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    flipud  = 0.0,
    fliplr  = 0.5,
    mosaic  = 1.0,
)

In [ ]:
# ── Validate ────────────────────────────────────────────────────
best_weights = f'/kaggle/working/runs/{PROJECT_NAME}/weights/best.pt'
model_best = YOLO(best_weights)
metrics = model_best.val(data=data_yaml, imgsz=IMG_SIZE)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

In [ ]:
# ── Quick inference check ────────────────────────────────────────
import glob, random
from IPython.display import Image as IPImage, display

sample_images = glob.glob(os.path.join(DATASET_PATH, 'val/images/*.jpg'))[:3]
if sample_images:
    img = random.choice(sample_images)
    res = model_best.predict(img, save=True, project='/kaggle/working/samples', name='check')
    saved = glob.glob('/kaggle/working/samples/check/*.jpg')
    if saved:
        display(IPImage(saved[0]))
else:
    print('No validation images found — check DATASET_PATH')

In [ ]:
# ── Copy weights to /kaggle/working so you can download them ────
output_path = '/kaggle/working/model1_vehicle_detection.pt'
shutil.copy(best_weights, output_path)
print(f'Model saved to: {output_path}')
print('Download it from the Kaggle Output tab.')